# Task 2 – HAM vs. SPAM: Vocabulary Differences

Which words are distinctively associated with HAM or SPAM?

## Method: Log-Odds Ratio

For each word we compute:

$$\text{log-odds}(w) = \log \frac{P(w \mid \text{SPAM})}{P(w \mid \text{HAM})}$$

- **Positive** → word leans towards SPAM
- **Negative** → word leans towards HAM
- **Laplace smoothing** (add-one) prevents division by zero for unseen words

Unlike raw counts, log-odds corrects for class imbalance (86 % HAM / 14 % SPAM).

In [1]:
from pathlib import Path

from sms_analysis import load_dataset
from sms_analysis.analysis import log_odds_ratio
from sms_analysis.visualization import plot_ham_spam_comparison, save_figure

DATASET = Path("../assignment/3D8026C4.zip/SMSSpamCollection")
OUTPUTS = Path("../outputs")

dataset = load_dataset(DATASET)
ham_texts  = [m.text for m in dataset.ham]
spam_texts = [m.text for m in dataset.spam]

In [2]:
df = log_odds_ratio(ham_texts, spam_texts, top_n=20)

print("Top SPAM words:")
print(df[df.leaning == "spam"][["word", "log_odds", "spam_count"]].to_string(index=False))
print("\nTop HAM words:")
print(df[df.leaning == "ham"][["word", "log_odds", "ham_count"]].to_string(index=False))

Top SPAM words:
      word  log_odds  spam_count
     claim  5.562510         113
      £num  5.456174         204
     prize  5.358911          92
guaranteed  4.758137          50
      tone  4.718132          48
   awarded  4.489873          38
    150ppm  4.381659          34
       tcs  4.227509          29
 numnumnum  4.158516          27
  ringtone  4.122148          26
     entry  4.122148          26
collection  4.122148          26
     tones  4.084408          25
       url  4.064990         101
    weekly  4.045187          24
       mob  4.045187          24
      150p  4.004365          23
     valid  4.004365          23
       sae  3.917354          21
     bonus  3.917354          21

Top HAM words:
     word  log_odds  ham_count
        ü -4.309487        169
      lor -4.255093        160
    later -4.078964        134
       da -4.056491        131
      ill -3.957005        238
    thats -3.738037         95
      ask -3.662325         88
     said -3.651026        

In [3]:
fig = plot_ham_spam_comparison(df)
save_figure(fig, OUTPUTS / "02_ham_vs_spam_log_odds.png")

## Observations

**Strongly SPAM-leaning words** reveal the characteristic language of commercial/malicious SMS:
- Action triggers: `claim`, `win`, `prize`, `free`, `urgent`
- Contact channels: `call`, `txt`, `mobile`, `__url__`
- Financial bait: `cash`, `award`, `£`, `guaranteed`

**Strongly HAM-leaning words** reflect natural conversational language:
- Personal pronouns and informality: `im`, `ur`, `wat`, `da`
- Relational terms: `home`, `morning`, `dinner`, `later`
- SMS-specific abbreviations not adopted by spammers

The asymmetry is striking: SPAM vocabulary is highly formulaic and marketer-driven,
while HAM vocabulary is varied and personal. This is why even simple linear models
achieve very high classification accuracy on this dataset.